# Stage 1 + M7B analysis

**Run this notebook locally — no GPU required.**

Loads every JSON artefact under `training/runs/*/` and regenerates every plot used in `blog.md`. If you re-run an ablation in Colab and drop the new JSON files into a new `training/runs/<name>/` folder, just re-execute this notebook to refresh all comparison charts.

Sections:
1. Discover all runs → results matrix
2. Per-run plots (price efficiency, signal alignment, P&L, direction inference, dashboard)
3. Cross-run comparison (M7B ablation table + grouped probe chart)

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from training import plots
from training.results_matrix import (
    add_baseline_rows,
    collect_rows,
    render_markdown,
)

RUNS_ROOT = REPO_ROOT / 'training' / 'runs'
print('runs root:', RUNS_ROOT)
for p in sorted(RUNS_ROOT.iterdir()) if RUNS_ROOT.exists() else []:
    print(' -', p.name)

## 1 — Results matrix

Walks every `training/runs/<name>/` folder that contains an `eval_trained.json` and assembles a single comparison row. Scripted baselines (`random`, `hold`) are pulled in once from any run that has them.

In [ ]:
rows = collect_rows(RUNS_ROOT)
rows = add_baseline_rows(rows, RUNS_ROOT)
row_dicts = [r.__dict__ for r in rows]
print(render_markdown(rows))

## 2 — Per-run plots

For each run that has full ToM probe coverage, regenerate all five plots.

In [ ]:
def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8')) if Path(path).exists() else None

def regenerate_for_run(run_dir, save=True):
    eval_trained = load_json(run_dir / 'eval_trained.json')
    eval_random = load_json(run_dir / 'eval_random.json')
    eval_hold = load_json(run_dir / 'eval_hold.json')
    baselines = load_json(run_dir / 'tom_probes_baselines.json') or {}
    trained_probes = load_json(run_dir / 'tom_probes_trained.json') or {}
    direction = load_json(run_dir / 'tom_direction_inference.json')
    trained_block = trained_probes.get('trained') if trained_probes else None

    figs = {}
    if baselines:
        figs['tom_price_efficiency.png'] = plots.plot_price_efficiency(baselines, trained_block)
        figs['tom_signal_alignment.png'] = plots.plot_signal_alignment(baselines, trained_block)
    policies = {k: v for k, v in {'hold': eval_hold, 'random': eval_random, 'trained': eval_trained}.items() if v}
    if policies:
        figs['eval_pnl_comparison.png'] = plots.plot_pnl_comparison(policies)
    if direction:
        figs['tom_direction_inference.png'] = plots.plot_direction_inference(direction)
    if baselines and policies and direction:
        figs['m6_summary_dashboard.png'] = plots.plot_summary_dashboard(
            baselines, trained_block, policies, direction,
        )
    if save:
        for name, fig in figs.items():
            fig.savefig(run_dir / name, dpi=150)
            print(f'  wrote {run_dir.name}/{name}')
    return figs

for run_dir in sorted(p for p in RUNS_ROOT.iterdir() if p.is_dir()) if RUNS_ROOT.exists() else []:
    if not (run_dir / 'eval_trained.json').exists():
        continue
    print(f'\n[{run_dir.name}]')
    regenerate_for_run(run_dir, save=True)
    plt.close('all')

## 3 — Cross-run comparison (M7B ablation)

Stays empty until at least one ablation is added under `training/runs/`.

In [ ]:
fig = plots.plot_pnl_matrix(row_dicts)
fig.savefig(RUNS_ROOT / 'pnl_matrix.png', dpi=150)
fig

In [ ]:
fig = plots.plot_probe_matrix(row_dicts)
fig.savefig(RUNS_ROOT / 'probe_matrix.png', dpi=150)
fig

## 4 — Save the matrix as JSON + Markdown

These two artefacts are what the blog references.

In [ ]:
from training.results_matrix import save_matrix

save_matrix(rows, RUNS_ROOT / 'results_matrix.json')
(RUNS_ROOT / 'results_matrix.md').write_text(render_markdown(rows), encoding='utf-8')
print('  wrote results_matrix.json and results_matrix.md')